In [3]:
import os
import pandas as pd

data = []  # Haber metinlerini ve etiketleri tutacak liste

# Ana klasörü belirt
root_dir = "42bin_haber/news"  # Eğer "news" klasörü burada ise, tam yolunu yaz

# Kategorilere göre klasörleri dolaş
for category in os.listdir(root_dir):
    category_path = os.path.join(root_dir, category)
    
    # Klasör mü kontrol et
    if os.path.isdir(category_path):
        # Klasördeki tüm .txt dosyalarını oku
        for filename in os.listdir(category_path):
            file_path = os.path.join(category_path, filename)
            
            # Dosya olup olmadığını kontrol et
            if os.path.isfile(file_path) and filename.endswith(".txt"):
                with open(file_path, "r", encoding="utf-8") as file:
                    content = file.read().replace("\n", " ").strip()  # Satır sonlarını temizle
                    
                    # DataFrame için veri ekle
                    data.append((content, category))

# Pandas DataFrame oluştur
df = pd.DataFrame(data, columns=["text", "label"])

# İlk 5 veriyi göster
print(df.head())

                                                text  label
0  'Ortak vizyonumuz var' Dışişleri Bakanı Davuto...  dunya
1  İsrail'den Gazze Şeridi'ne hava saldırısı İsra...  dunya
2  Cenaze için geniş güvenlik önlemleri alındı Lü...  dunya
3  Gözaltındaki sendikacılar serbest KKTC'de Send...  dunya
4  Bisikletle Asya'da 3 bin kilometre yol katetti...  dunya


In [4]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords

# İlk olarak, orijinal DataFrame'in bir kopyasını oluşturalım
df_cleaned = df.copy()

# Türkçe stopword'leri indirme
nltk.download("stopwords")
stop_words = set(stopwords.words("turkish"))

# Temizleme fonksiyonu
def clean_text(text):
    text = text.lower()  # Küçük harfe çevir
    text = re.sub(f"[{string.punctuation}]", " ", text)  # Noktalama işaretlerini kaldır
    text = re.sub(r"\d+", " ", text)  # Sayıları kaldır
    text = " ".join([word for word in text.split() if word not in stop_words])  # Stopword'leri kaldır
    text = re.sub(r"\s+", " ", text).strip()  # Fazla boşlukları temizle
    return text

# Temizleme işlemini uygula
df_cleaned["text"] = df_cleaned["text"].apply(clean_text)

# İlk 5 veriyi göster
print(df_cleaned.head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ertuğrul\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


                                                text  label
0  ortak vizyonumuz var dışişleri bakanı davutoğl...  dunya
1  i̇srail den gazze şeridi hava saldırısı i̇srai...  dunya
2  cenaze geniş güvenlik önlemleri alındı lübnan ...  dunya
3  gözaltındaki sendikacılar serbest kktc sendika...  dunya
4  bisikletle asya bin kilometre yol katettiler t...  dunya


In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# df_cleaned veri seti zaten daha önce temizlenmiş ve işlenmiş durumda

# Metni ve etiketleri ayır
X = df_cleaned['text']  # Metin sütunu
y = df_cleaned['label']  # Etiket (label) sütunu

# Eğitim ve test verisi olarak ayır
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TF-IDF Vectorizer ile metni sayısal verilere dönüştür
tfidf = TfidfVectorizer(max_features=5000)  
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Lojistik Regresyon sınıflandırıcısını tanımla ve eğit
clf = LogisticRegression(max_iter=1000)  # max_iter yüksek tutarak optimizasyonun tamamlanmasını sağlıyoruz
clf.fit(X_train_tfidf, y_train)

# Test verisiyle tahmin yap
y_pred = clf.predict(X_test_tfidf)

# Sonuçları değerlendir
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}')
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.6384

Classification Report:
              precision    recall  f1-score   support

       dunya       0.63      0.73      0.68       789
     ekonomi       0.66      0.71      0.68       675
       genel       0.34      0.30      0.32      1319
      guncel       0.49      0.63      0.55      1121
kultur-sanat       0.70      0.60      0.65       250
     magazin       0.69      0.74      0.71       562
      planet       0.56      0.34      0.42       361
      saglik       0.75      0.75      0.75       243
     siyaset       0.51      0.50      0.50       365
        spor       0.90      0.98      0.94      2039
   teknoloji       0.67      0.54      0.60       159
     turkiye       0.50      0.18      0.26       387
       yasam       0.83      0.04      0.07       129

    accuracy                           0.64      8399
   macro avg       0.63      0.54      0.55      8399
weighted avg       0.63      0.64      0.62      8399



In [6]:
# Yeni bir metin girişi yapalım ve tahmin edelim
new_text = ["Galatasar derbiyi kazanıp şampiyonluğunu ilan etti."]  # Buraya yeni metninizi yazın

# Yeni metni temizle (ön işleme işlemi)
new_text_cleaned = [clean_text(text) for text in new_text]

# Yeni metni TF-IDF ile dönüştür
new_text_tfidf = tfidf.transform(new_text_cleaned)

# Tahmin yap
predicted_label = clf.predict(new_text_tfidf)

# Tahmin edilen etiket
print(f"Yeni metnin tahmin edilen etiketi: {predicted_label[0]}")

Yeni metnin tahmin edilen etiketi: spor


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# RandomForest sınıflandırıcıyı oluştur
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)

# Modeli eğitim verisi ile eğit
rf_clf.fit(X_train_tfidf, y_train)

# Eğitim verisi üzerinde tahmin yap
y_pred_rf = rf_clf.predict(X_test_tfidf)

# Sonuçları değerlendirelim
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nRandom Forest Classification Report:\n", classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.6190022621740684

Random Forest Classification Report:
               precision    recall  f1-score   support

       dunya       0.58      0.68      0.63       789
     ekonomi       0.64      0.72      0.68       675
       genel       0.32      0.28      0.30      1319
      guncel       0.47      0.61      0.53      1121
kultur-sanat       0.75      0.52      0.61       250
     magazin       0.64      0.79      0.71       562
      planet       0.64      0.20      0.31       361
      saglik       0.73      0.73      0.73       243
     siyaset       0.52      0.47      0.49       365
        spor       0.88      0.98      0.93      2039
   teknoloji       0.74      0.35      0.48       159
     turkiye       0.38      0.16      0.23       387
       yasam       0.33      0.07      0.12       129

    accuracy                           0.62      8399
   macro avg       0.59      0.50      0.52      8399
weighted avg       0.60      0.62      0.60      839

In [8]:
# Yeni bir metin girişi yapalım ve tahmin edelim
new_text = ["Galatasar derbiyi kazanıp şampiyonluğunu ilan etti."]  # Buraya yeni metninizi yazın

# Yeni metni temizle (ön işleme işlemi)
new_text_cleaned = [clean_text(text) for text in new_text]

# Yeni metni TF-IDF ile dönüştür
new_text_tfidf = tfidf.transform(new_text_cleaned)

# Tahmin yap
predicted_label = clf.predict(new_text_tfidf)

# Tahmin edilen etiket
print(f"Yeni metnin tahmin edilen etiketi: {predicted_label[0]}")

Yeni metnin tahmin edilen etiketi: spor
